In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1TJ98zZz2KklANHNy7XACiJpE_9AGhpWM", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_01_intro.mp3"))

# Validation-Retry Loops

**Notebook 3 of 4** | Prompt Engineering & Structured Output | Claude Certified Architect Prep

---

**Objective:** Build extract-validate-retry pipelines that automatically catch and correct errors, implement error feedback for self-correction, and learn to identify when retries are futile.

**Exam Task Covered:** 4.4 — validation-retry loops, retry-with-error-feedback

**Prerequisites:** Notebooks 01 (Explicit Criteria & Few-Shot) and 02 (Structured Output with tool_use)

---

In the last notebook we built tool_use schemas that force Claude to return structured JSON. That works beautifully—most of the time. In this notebook, we tackle the hard follow-up question: **what happens when the structured output is syntactically valid but semantically wrong?**

By the end of this notebook, you will be able to:
1. Write validation functions that catch math, format, and business-rule errors
2. Implement the retry-with-error-feedback pattern (the exact pattern tested on the exam)
3. Classify errors as retryable vs. futile to avoid wasting API calls
4. Track extraction patterns to systematically improve your prompts over time

In [ ]:
#@title 🎧 Listen: 92 Percent Problem
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_92_percent_problem.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Retryable Distinction
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_retryable_distinction.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Concept Overview

### The 92% Problem

Here's a number that comes up a lot in production extraction systems: **~92% of tool_use calls return correct, valid data on the first try**. That sounds great—until you realize the other 8% can silently corrupt your database, miscalculate financial totals, or break downstream systems.

The schema from Notebook 02 guarantees *structure* (the JSON has the right keys and types). But it cannot guarantee *correctness*:
- Line items that don't sum to the total
- Dates formatted correctly but logically impossible (invoice dated next year)
- Fields that pass type-checking but violate business rules

### The Retry-with-Error-Feedback Pattern

The pattern is simple but powerful:

```
1. EXTRACT  →  Call Claude with tool_use, get structured output
2. VALIDATE →  Run domain-specific checks on the extracted data  
3. RETRY    →  If validation fails, feed the SPECIFIC errors back to Claude
```

The key insight: **we don't just say "try again."** We tell Claude exactly what went wrong:

> *"Your extraction had 2 errors: (1) Line items sum to $450.00 but total_amount is $500.00. (2) invoice_date 2027-03-15 is in the future."*

This targeted feedback lets Claude fix the specific issues without re-introducing new ones.

### Critical Distinction: Retries Fix Format Errors, Not Missing Information

This is the single most important thing to internalize for the exam:

| Error Type | Retryable? | Example |
|---|---|---|
| Math/calculation errors | YES | Line items don't sum correctly |
| Format errors | YES | Date is `03/15/2025` instead of `2025-03-15` |
| Missing information | **NO** | `vendor_name` is null because the document has no vendor name |
| Ambiguous source | RISKY | Two possible invoice numbers in the document |

Retrying a missing-information error just wastes tokens. If the vendor name isn't in the document, Claude can't conjure it from thin air—and if it does, that's a hallucination. Use **nullable fields** instead.

### The `detected_pattern` Field

Smart extraction systems don't just fix errors—they track them. By logging a `detected_pattern` string on each failure (e.g., `"math_error_line_items"`, `"date_format_mismatch"`), you can identify systematic issues and add targeted few-shot examples to prevent them upstream. We will build this tracking system in Section 4.

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Setup

We will use a mock client that lets us pre-configure sequences of responses—so we can test retry behavior deterministically. In production, you would swap this for `anthropic.Anthropic()`.

In [ ]:
!pip install anthropic -q

import json
import re
from datetime import datetime, date
from typing import Any


class RetryMockClient:
    """Mock Claude client that returns pre-configured responses.
    
    This lets us test retry logic deterministically without API calls.
    In production, replace with: client = anthropic.Anthropic()
    """
    
    def __init__(self):
        self.call_count = 0
        self.responses = []
        self.messages = type('Messages', (), {'create': self._create})()
    
    def set_responses(self, responses: list[dict]):
        """Pre-configure a sequence of responses for testing.
        
        Args:
            responses: List of dicts. Each dict is the tool_use input
                       that Claude would return on that attempt.
        """
        self.responses = responses
        self.call_count = 0
    
    def _create(self, model="claude-sonnet-4-20250514", max_tokens=1024,
                tools=None, tool_choice=None, messages=None, **kwargs):
        """Simulate a Claude API call that returns tool_use."""
        idx = min(self.call_count, len(self.responses) - 1)
        self.call_count += 1
        response_data = self.responses[idx]
        
        tool_name = tools[0]["name"] if tools else "extract"
        
        # Build a response object that mimics the real API
        tool_block = type('ToolUseBlock', (), {
            'type': 'tool_use',
            'name': tool_name,
            'input': response_data,
            'id': f'toolu_{self.call_count:04d}'
        })()
        
        return type('Response', (), {
            'content': [tool_block],
            'stop_reason': 'tool_use'
        })()


client = RetryMockClient()
print("Setup complete! Mock client ready for testing retry behavior.")

In [ ]:
#@title 🎧 Listen: Validation Engine
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_validation_engine.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 1: Building a Validation Engine

Before we can retry, we need to know *what's wrong*. A good validation engine checks four layers:

| Layer | What It Catches | Example |
|---|---|---|
| **Schema** | Wrong types, missing required fields | `total_amount` is a string instead of float |
| **Mathematical** | Arithmetic inconsistencies | Line items don't sum to total |
| **Logical** | Impossible values | Date in the future, negative quantity |
| **Business Rules** | Domain-specific constraints | Invoice number doesn't match company format |

The schema layer is already handled by `tool_use` (if a field is `number`, Claude must return a number). Our validation engine focuses on layers 2–4: the semantic checks that schemas can't express.

Let's build a validator for invoice extraction—one of the most common real-world extraction tasks.

In [ ]:
#@title 🎧 Listen: Todo1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_todo1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ============================================================
# TODO 1: Build a validate_invoice() function
# ============================================================
#
# Given an extracted invoice dict, check ALL of the following:
#
# 1. MATH CHECK: Each line item's qty * unit_price should equal
#    line_total (within $0.01 tolerance)
#
# 2. MATH CHECK: Sum of all line_total values should equal
#    total_amount (within $0.01 tolerance)
#
# 3. FORMAT CHECK: vendor_name must not be empty or None
#
# 4. FORMAT CHECK: invoice_number must match pattern INV-XXXX
#    (INV- followed by exactly 4 digits)
#
# 5. LOGICAL CHECK: invoice_date must not be in the future
#
# Return: list[str] of error messages (empty list = valid)
#
# Invoice structure:
# {
#   "vendor_name": str,
#   "invoice_number": str,
#   "invoice_date": str (YYYY-MM-DD),
#   "total_amount": float,
#   "line_items": [
#     {"description": str, "qty": int, "unit_price": float, "line_total": float}
#   ]
# }
# ============================================================

def validate_invoice(invoice: dict) -> list[str]:
    """Validate an extracted invoice for math, format, and logical errors."""
    errors = []
    
    # --- YOUR CODE HERE ---
    pass
    # --- END YOUR CODE ---
    
    return errors


# Quick smoke test (don't modify)
print("Function defined. Run the test cell below to check your work.")

In [ ]:
#@title 🎧 Listen: Todo1 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_todo1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ==========================
# SOLUTION (TODO 1)
# ==========================

def validate_invoice(invoice: dict) -> list[str]:
    """Validate an extracted invoice for math, format, and logical errors.
    
    Returns a list of human-readable error strings.
    An empty list means the invoice is valid.
    """
    errors = []
    
    # 1. MATH CHECK: Each line item qty * unit_price == line_total
    for i, item in enumerate(invoice.get("line_items", [])):
        expected = item["qty"] * item["unit_price"]
        actual = item["line_total"]
        if abs(expected - actual) > 0.01:
            errors.append(
                f"Line item {i+1} ({item['description']}): "
                f"qty({item['qty']}) * unit_price(${item['unit_price']:.2f}) = "
                f"${expected:.2f}, but line_total is ${actual:.2f}"
            )
    
    # 2. MATH CHECK: Sum of line_totals == total_amount
    line_sum = sum(item["line_total"] for item in invoice.get("line_items", []))
    total = invoice.get("total_amount", 0)
    if abs(line_sum - total) > 0.01:
        errors.append(
            f"Line items sum to ${line_sum:.2f} but total_amount is ${total:.2f}"
        )
    
    # 3. FORMAT CHECK: vendor_name must not be empty
    vendor = invoice.get("vendor_name")
    if not vendor or not vendor.strip():
        errors.append("vendor_name is empty or missing")
    
    # 4. FORMAT CHECK: invoice_number must match INV-XXXX
    inv_num = invoice.get("invoice_number", "")
    if not re.match(r'^INV-\d{4}$', inv_num):
        errors.append(
            f"invoice_number '{inv_num}' does not match required pattern INV-XXXX"
        )
    
    # 5. LOGICAL CHECK: invoice_date must not be in the future
    try:
        inv_date = datetime.strptime(invoice.get("invoice_date", ""), "%Y-%m-%d").date()
        if inv_date > date.today():
            errors.append(
                f"invoice_date {invoice['invoice_date']} is in the future"
            )
    except ValueError:
        errors.append(
            f"invoice_date '{invoice.get('invoice_date', '')}' is not a valid YYYY-MM-DD date"
        )
    
    return errors


print("Solution loaded.")

In [ ]:
# ============================================================
# Test Cases for validate_invoice()
# ============================================================

# Test 1: Valid invoice (should return NO errors)
valid_invoice = {
    "vendor_name": "Acme Corp",
    "invoice_number": "INV-1234",
    "invoice_date": "2025-01-15",
    "total_amount": 250.00,
    "line_items": [
        {"description": "Widget A", "qty": 5, "unit_price": 30.00, "line_total": 150.00},
        {"description": "Widget B", "qty": 2, "unit_price": 50.00, "line_total": 100.00}
    ]
}

errors = validate_invoice(valid_invoice)
print("Test 1 - Valid invoice:")
print(f"  Errors: {errors}")
assert len(errors) == 0, f"Expected 0 errors, got {len(errors)}"
print("  PASSED\n")

# Test 2: Math errors (line item wrong + total wrong)
math_error_invoice = {
    "vendor_name": "Acme Corp",
    "invoice_number": "INV-5678",
    "invoice_date": "2025-02-10",
    "total_amount": 500.00,  # Wrong! Should be 250.00
    "line_items": [
        {"description": "Widget A", "qty": 5, "unit_price": 30.00, "line_total": 200.00},  # Wrong! 5*30=150
        {"description": "Widget B", "qty": 2, "unit_price": 50.00, "line_total": 100.00}
    ]
}

errors = validate_invoice(math_error_invoice)
print("Test 2 - Math errors:")
for e in errors:
    print(f"  - {e}")
assert len(errors) == 2, f"Expected 2 errors, got {len(errors)}"
print("  PASSED\n")

# Test 3: Missing vendor + bad invoice number + future date
format_error_invoice = {
    "vendor_name": "",
    "invoice_number": "INVOICE-1234",
    "invoice_date": "2099-01-01",
    "total_amount": 100.00,
    "line_items": [
        {"description": "Service", "qty": 1, "unit_price": 100.00, "line_total": 100.00}
    ]
}

errors = validate_invoice(format_error_invoice)
print("Test 3 - Format/logical errors:")
for e in errors:
    print(f"  - {e}")
assert len(errors) == 3, f"Expected 3 errors, got {len(errors)}"
print("  PASSED\n")

print("All validation tests passed!")

In [ ]:
#@title 🎧 Listen: Retry Pattern
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_retry_pattern.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 2: The Retry-with-Error-Feedback Pattern

Now that we can detect errors, let's build the retry loop. The key is **how** we feed errors back. Here is the message chain that makes this work:

```
Message 1 (user):     "Extract the invoice from this document: ..."
Message 2 (assistant): [tool_use with bad extraction]
Message 3 (user):     "Your extraction had errors:\n1. Line items sum to $450 but total is $500\nPlease re-extract with corrections."
Message 4 (assistant): [tool_use with corrected extraction]
```

Notice that we include the **assistant's failed attempt** in the conversation. This is critical—Claude needs to see what it got wrong to fix it. If you start a fresh conversation, Claude has no context about its previous mistake and may repeat it.

### Why This Works

When Claude sees its own incorrect output alongside specific error messages, it can:
1. **Identify the exact fields** that need correction
2. **Preserve correct fields** (it doesn't start from scratch)
3. **Apply the correction** with the original document still in context

This is dramatically better than a blind retry, which might fix one error but introduce another.

In [ ]:
# First, let's define our invoice extraction tool schema
# (same pattern as Notebook 02, but now we'll validate the output)

INVOICE_TOOL = {
    "name": "extract_invoice",
    "description": "Extract structured invoice data from a document.",
    "input_schema": {
        "type": "object",
        "properties": {
            "vendor_name": {
                "type": "string",
                "description": "Name of the vendor/supplier"
            },
            "invoice_number": {
                "type": "string",
                "description": "Invoice number in format INV-XXXX"
            },
            "invoice_date": {
                "type": "string",
                "description": "Invoice date in YYYY-MM-DD format"
            },
            "total_amount": {
                "type": "number",
                "description": "Total invoice amount in dollars"
            },
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "qty": {"type": "integer"},
                        "unit_price": {"type": "number"},
                        "line_total": {"type": "number"}
                    },
                    "required": ["description", "qty", "unit_price", "line_total"]
                },
                "description": "Individual line items on the invoice"
            }
        },
        "required": ["vendor_name", "invoice_number", "invoice_date", "total_amount", "line_items"]
    }
}

print("Invoice tool schema defined.")
print(f"Required fields: {INVOICE_TOOL['input_schema']['required']}")

In [ ]:
#@title 🎧 Listen: Todo2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_todo2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ============================================================
# TODO 2: Implement extract_with_retry()
# ============================================================
#
# Build the core retry-with-error-feedback loop:
#
# 1. Create initial messages list with the user's document
# 2. Call client.messages.create() with the invoice tool
# 3. Extract the tool_use result from the response
# 4. Validate using validate_invoice()
# 5. If valid: return immediately with success
# 6. If errors AND retries remaining:
#    a. Append the assistant's response to messages
#    b. Append a user message with the specific errors
#    c. Loop back to step 2
# 7. If errors AND no retries left: return with the errors
#
# Return format:
# {
#   "data": dict,           # The extracted invoice (best attempt)
#   "attempts": int,        # Number of API calls made
#   "errors": list[str],    # Empty if valid, otherwise last errors
#   "success": bool         # True if final result passed validation
# }
#
# IMPORTANT: The error feedback message should look like:
#   "Your extraction had {n} validation error(s):\n"
#   "1. {error_1}\n2. {error_2}\n..."
#   "Please re-extract the invoice with these errors corrected."
# ============================================================

def extract_with_retry(document: str, max_retries: int = 2) -> dict:
    """Extract invoice data with validation and retry-with-error-feedback."""
    
    # --- YOUR CODE HERE ---
    pass
    # --- END YOUR CODE ---


print("Function defined. Run the test cell below to check your work.")

In [ ]:
#@title 🎧 Listen: Todo2 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_todo2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ==========================
# SOLUTION (TODO 2)
# ==========================

def extract_with_retry(document: str, max_retries: int = 2) -> dict:
    """Extract invoice data with validation and retry-with-error-feedback.
    
    Uses the retry-with-error-feedback pattern:
    - Extract via tool_use
    - Validate the result
    - If errors: append failed attempt + specific errors to messages
    - Retry until valid or max_retries exhausted
    """
    messages = [
        {
            "role": "user",
            "content": f"Extract the invoice data from this document:\n\n{document}"
        }
    ]
    
    attempts = 0
    last_errors = []
    last_data = None
    
    for attempt in range(1 + max_retries):  # 1 initial + max_retries
        attempts += 1
        
        # Call the API
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=[INVOICE_TOOL],
            tool_choice={"type": "tool", "name": "extract_invoice"},
            messages=messages
        )
        
        # Extract tool_use result
        tool_block = response.content[0]
        extracted = tool_block.input
        last_data = extracted
        
        # Validate
        errors = validate_invoice(extracted)
        
        if not errors:
            # Valid! Return success.
            return {
                "data": extracted,
                "attempts": attempts,
                "errors": [],
                "success": True
            }
        
        last_errors = errors
        
        # If we have retries left, build the error feedback
        if attempt < max_retries:
            # Append the assistant's failed response
            messages.append({
                "role": "assistant",
                "content": [{
                    "type": "tool_use",
                    "id": tool_block.id,
                    "name": tool_block.name,
                    "input": extracted
                }]
            })
            
            # Build error feedback message
            error_list = "\n".join(
                f"{i+1}. {err}" for i, err in enumerate(errors)
            )
            feedback = (
                f"Your extraction had {len(errors)} validation error(s):\n"
                f"{error_list}\n"
                f"Please re-extract the invoice with these errors corrected."
            )
            
            # Append as tool_result (required after tool_use) + user message
            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_block.id,
                        "content": feedback
                    }
                ]
            })
    
    # Exhausted all retries
    return {
        "data": last_data,
        "attempts": attempts,
        "errors": last_errors,
        "success": False
    }


print("Solution loaded.")

In [ ]:
#@title 🎧 Listen: Todo3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_todo3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ============================================================
# TODO 3: Test the retry function with mock responses
# ============================================================
#
# Configure the mock client with pre-defined response sequences
# and verify the retry behavior works correctly.
#
# Test 1: First extraction is correct -> should succeed on attempt 1
# Test 2: First extraction has a math error, second is correct -> succeed on attempt 2
# Test 3: All 3 attempts have errors -> should return failure with errors
#
# For each test:
#   1. Call client.set_responses([...]) with the mock data
#   2. Call extract_with_retry(document)
#   3. Print and verify the result
# ============================================================

test_document = """INVOICE
Vendor: Acme Corp
Invoice #: INV-4321
Date: 2025-03-01

Items:
- 10x Bolts @ $2.50 each = $25.00
- 5x Nuts @ $1.00 each = $5.00

Total: $30.00
"""

# --- YOUR CODE HERE ---

# Test 1: First-attempt success

# Test 2: Retry success

# Test 3: All attempts fail

# --- END YOUR CODE ---

In [ ]:
#@title 🎧 Listen: Todo3 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_todo3_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ==========================
# SOLUTION (TODO 3)
# ==========================

test_document = """INVOICE
Vendor: Acme Corp
Invoice #: INV-4321
Date: 2025-03-01

Items:
- 10x Bolts @ $2.50 each = $25.00
- 5x Nuts @ $1.00 each = $5.00

Total: $30.00
"""

# ---- Test 1: First-attempt success ----
print("=" * 50)
print("TEST 1: First-attempt success")
print("=" * 50)

client.set_responses([
    {  # Attempt 1: correct
        "vendor_name": "Acme Corp",
        "invoice_number": "INV-4321",
        "invoice_date": "2025-03-01",
        "total_amount": 30.00,
        "line_items": [
            {"description": "Bolts", "qty": 10, "unit_price": 2.50, "line_total": 25.00},
            {"description": "Nuts", "qty": 5, "unit_price": 1.00, "line_total": 5.00}
        ]
    }
])

result = extract_with_retry(test_document)
print(f"  Success: {result['success']}")
print(f"  Attempts: {result['attempts']}")
print(f"  Errors: {result['errors']}")
assert result["success"] == True
assert result["attempts"] == 1
print("  PASSED\n")

# ---- Test 2: Retry success (math error on attempt 1, correct on attempt 2) ----
print("=" * 50)
print("TEST 2: Retry success (fixed on second attempt)")
print("=" * 50)

client.set_responses([
    {  # Attempt 1: math error (total is wrong)
        "vendor_name": "Acme Corp",
        "invoice_number": "INV-4321",
        "invoice_date": "2025-03-01",
        "total_amount": 50.00,  # WRONG: should be 30.00
        "line_items": [
            {"description": "Bolts", "qty": 10, "unit_price": 2.50, "line_total": 25.00},
            {"description": "Nuts", "qty": 5, "unit_price": 1.00, "line_total": 5.00}
        ]
    },
    {  # Attempt 2: corrected
        "vendor_name": "Acme Corp",
        "invoice_number": "INV-4321",
        "invoice_date": "2025-03-01",
        "total_amount": 30.00,
        "line_items": [
            {"description": "Bolts", "qty": 10, "unit_price": 2.50, "line_total": 25.00},
            {"description": "Nuts", "qty": 5, "unit_price": 1.00, "line_total": 5.00}
        ]
    }
])

result = extract_with_retry(test_document)
print(f"  Success: {result['success']}")
print(f"  Attempts: {result['attempts']}")
print(f"  Errors: {result['errors']}")
assert result["success"] == True
assert result["attempts"] == 2
print("  PASSED\n")

# ---- Test 3: All attempts fail ----
print("=" * 50)
print("TEST 3: All attempts fail (exhausted retries)")
print("=" * 50)

bad_response = {
    "vendor_name": "Acme Corp",
    "invoice_number": "INV-4321",
    "invoice_date": "2025-03-01",
    "total_amount": 999.99,  # WRONG every time
    "line_items": [
        {"description": "Bolts", "qty": 10, "unit_price": 2.50, "line_total": 25.00},
        {"description": "Nuts", "qty": 5, "unit_price": 1.00, "line_total": 5.00}
    ]
}

client.set_responses([bad_response, bad_response, bad_response])

result = extract_with_retry(test_document)
print(f"  Success: {result['success']}")
print(f"  Attempts: {result['attempts']}")
print(f"  Errors: {result['errors']}")
assert result["success"] == False
assert result["attempts"] == 3  # 1 initial + 2 retries
assert len(result["errors"]) > 0
print("  PASSED\n")

print("All retry tests passed!")

In [ ]:
#@title 🎧 Listen: When Retries Futile
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_when_retries_futile.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 3: When Retries Are Futile

This section covers one of the most important judgment calls in production extraction systems: **knowing when NOT to retry.**

Retries cost time and tokens. More importantly, retrying a fundamentally un-fixable error can produce **hallucinations**—Claude might invent a vendor name that doesn't exist in the document just to satisfy the validation constraint.

### Three Categories of Extraction Failures

**1. Format/Math Errors → RETRYABLE**

The model had all the information it needed—it just processed it incorrectly.

Examples:
- Line items don't sum to total (arithmetic mistake)
- Date formatted as `03/15/2025` instead of `2025-03-15`
- Dollar amount parsed as `"$1,234"` (string) instead of `1234.0` (number)

These are the best candidates for retry because the correction is deterministic.

**2. Missing Information → NOT RETRYABLE**

The information simply doesn't exist in the source document.

Examples:
- `vendor_name` is null because the receipt only has a logo, no text name
- `policy_number` is null because the claim form was incomplete
- `email` is null because the business card only has a phone number

Retrying this will either return null again (wasting tokens) or hallucinate a value (worse). The correct fix is to make the field **nullable** in your schema and handle `null` downstream.

**3. Ambiguous Source → RISKY**

Multiple valid interpretations exist in the source document.

Examples:
- Two different invoice numbers on the same page (original + reference)
- Amount says "approximately $12,000-$13,000" (which number do you pick?)
- Date could be `01/02/2025` (Jan 2 or Feb 1, depending on locale)

Retries may cause Claude to **flip-flop** between interpretations. Better to surface the ambiguity to a human reviewer.

In [ ]:
#@title 🎧 Listen: Todo4
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_todo4.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ============================================================
# TODO 4: Write a classify_error() function
# ============================================================
#
# Given a validation error message (string), classify it as:
#   - "retryable"  : format or math error (model had the info, processed wrong)
#   - "futile"     : missing information (data not in source document)
#   - "ambiguous"  : multiple valid interpretations exist
#
# Heuristics to use:
#   - If the error mentions "sum", "total", math symbols, or calculation
#     mismatches -> "retryable"
#   - If the error mentions "invalid" format, "does not match pattern",
#     or format issues -> "retryable"
#   - If the error mentions "null", "empty", "missing", "not found in
#     document" -> "futile"
#   - If the error mentions "two possible", "multiple", "ambiguous",
#     "approximately", or a range -> "ambiguous"
#   - Default to "retryable" if unsure
#
# Return: str (one of "retryable", "futile", "ambiguous")
# ============================================================

def classify_error(error_message: str) -> str:
    """Classify a validation error to decide if retry is worthwhile."""
    
    # --- YOUR CODE HERE ---
    pass
    # --- END YOUR CODE ---


print("Function defined.")

In [ ]:
# ==========================
# SOLUTION (TODO 4)
# ==========================

def classify_error(error_message: str) -> str:
    """Classify a validation error to decide if retry is worthwhile.
    
    Returns:
        'retryable'  - format/math error, retry should fix it
        'futile'     - missing info, retry won't help
        'ambiguous'  - multiple interpretations, risky to retry
    """
    msg = error_message.lower()
    
    # Check for missing information (futile) - check FIRST
    # because "is null" or "is empty" signals missing data
    futile_signals = [
        "is null", "is empty", "missing", "not found in document",
        "no .* found", "not in document", "not present"
    ]
    for signal in futile_signals:
        if re.search(signal, msg):
            return "futile"
    
    # Check for ambiguity
    ambiguous_signals = [
        "two possible", "multiple", "ambiguous",
        "approximately", "could be", r"\d+-\$?\d+",  # ranges like $12,000-$13,000
    ]
    for signal in ambiguous_signals:
        if re.search(signal, msg):
            return "ambiguous"
    
    # Check for math/format errors (retryable)
    retryable_signals = [
        "sum", "total", "does not match", "invalid",
        "format", "pattern", "expected", "but",
        "should be", "instead of"
    ]
    for signal in retryable_signals:
        if signal in msg:
            return "retryable"
    
    # Default: assume retryable (optimistic)
    return "retryable"


print("Solution loaded.")

In [ ]:
#@title 🎧 Listen: Todo5
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_todo5.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ============================================================
# TODO 5: Classify these 6 real-world validation errors
# ============================================================
#
# For each error, call classify_error() and explain WHY.
# After classifying all 6, decide: should we retry?
#
# Decision rule: Retry ONLY if ALL errors are retryable.
# If ANY error is futile or ambiguous, don't retry
# (handle those fields differently instead).
# ============================================================

test_errors = [
    "Line items sum to $450.00 but total_amount is $500.00",
    "vendor_name is empty — no vendor name found in document",
    "date_of_loss is 2025-15-03 (invalid month)",
    "policy_number is null — no policy number in document",
    "Two possible invoice numbers: INV-001 and REF-001",
    "claimed_amount is $12,500 but text says 'approximately $12,000-$13,000'",
]

# --- YOUR CODE HERE ---

# Classify each error and print the result
# Then make a retry decision

# --- END YOUR CODE ---

In [ ]:
# ==========================
# SOLUTION (TODO 5)
# ==========================

test_errors = [
    "Line items sum to $450.00 but total_amount is $500.00",
    "vendor_name is empty \u2014 no vendor name found in document",
    "date_of_loss is 2025-15-03 (invalid month)",
    "policy_number is null \u2014 no policy number in document",
    "Two possible invoice numbers: INV-001 and REF-001",
    "claimed_amount is $12,500 but text says 'approximately $12,000-$13,000'",
]

expected_classifications = [
    "retryable",   # Math error: model added wrong
    "futile",      # Missing info: vendor name not in document
    "retryable",   # Format error: month 15 is invalid, model misread
    "futile",      # Missing info: policy number not in document
    "ambiguous",   # Two valid values exist in the source
    "ambiguous",   # Source gives a range, not a precise number
]

explanations = [
    "Math/calculation error: model had line items but summed wrong",
    "Missing information: no vendor name exists in the source document",
    "Format error: model parsed the date digits in wrong order",
    "Missing information: policy number doesn't exist in the source",
    "Ambiguous: two different reference numbers in the document",
    "Ambiguous: source gives an approximate range, not exact value",
]

print("Error Classification Results")
print("=" * 60)

retryable_count = 0
futile_count = 0
ambiguous_count = 0

for i, error in enumerate(test_errors):
    classification = classify_error(error)
    emoji_map = {"retryable": "[RETRY]", "futile": "[SKIP]", "ambiguous": "[FLAG]"}
    
    print(f"\n{emoji_map[classification]} Error {i+1}: {error}")
    print(f"   Classification: {classification}")
    print(f"   Reason: {explanations[i]}")
    
    if classification == "retryable":
        retryable_count += 1
    elif classification == "futile":
        futile_count += 1
    else:
        ambiguous_count += 1

print("\n" + "=" * 60)
print(f"Summary: {retryable_count} retryable, {futile_count} futile, {ambiguous_count} ambiguous")

# Decision: should we retry?
if futile_count == 0 and ambiguous_count == 0:
    print("\nDecision: RETRY (all errors are retryable)")
else:
    print(f"\nDecision: DO NOT RETRY blindly.")
    print(f"  - {futile_count} errors require nullable fields (not retries)")
    print(f"  - {ambiguous_count} errors require human review")
    print(f"  - {retryable_count} errors could be fixed by retry")
    print(f"  Best approach: Retry ONLY the retryable errors, flag the rest.")

In [ ]:
#@title 🎧 Listen: Todo5 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_todo5_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Key Takeaway from This Exercise

Out of 6 errors, only 2 are worth retrying. Blindly retrying all 6 would:
1. **Waste 2 extra API calls** that can't fix the futile errors
2. **Risk hallucination** (Claude might invent a vendor name to stop the error)
3. **Not resolve ambiguity** (it might flip-flop between invoice numbers)

The smart approach: **classify errors first, then retry selectively**. This is exactly what we will build in the Integration Exercise.

In [ ]:
#@title 🎧 Listen: Tracking
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_tracking.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 4: Tracking Systematic Failures

Individual retries fix individual documents. But the real power move is **tracking patterns across extractions** to improve your system permanently.

### The `detected_pattern` Field

After each extraction (whether it succeeds or fails), we log metadata:

```python
{
    "document_id": "doc_0042",
    "attempts": 2,
    "success": True,
    "errors": ["Line items sum mismatch"],
    "detected_pattern": "math_error_line_items"  # <-- This is the key
}
```

The `detected_pattern` is a short, categorical label that groups similar failures. Over time, you can analyze these patterns:

- If **>30% of failures** share a pattern, add a few-shot example targeting that specific error
- If a pattern only appears with certain document types, you might need a specialized prompt
- If a pattern appears on retry attempts but not initial attempts, your error feedback is causing confusion

Let's build a tracker.

In [ ]:
#@title 🎧 Listen: Todo6
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_todo6.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# ============================================================
# TODO 6: Build an ExtractionTracker class
# ============================================================
#
# The tracker should:
#
# 1. record(document_id, attempts, success, errors, detected_pattern)
#    - Store each extraction result
#    - detected_pattern is a string like "math_error_line_items"
#      or "missing_vendor" or None (if no pattern detected)
#
# 2. summary() -> dict with:
#    - "total_extractions": int
#    - "success_count": int
#    - "failure_count": int  
#    - "success_rate": float (0-1)
#    - "avg_attempts": float
#    - "pattern_counts": dict mapping pattern -> count
#    - "recommendations": list[str] with actionable advice
#
# 3. Recommendation logic:
#    - If a pattern accounts for >30% of failures:
#      "Add few-shot example for pattern '{pattern}' ({count}/{failures} failures)"
#    - If average attempts > 1.5:
#      "High retry rate ({avg:.1f} avg attempts). Review prompt for systematic issues."
#    - If success rate < 0.8:
#      "Low success rate ({rate:.0%}). Consider adding more specific tool descriptions."
# ============================================================

class ExtractionTracker:
    """Track extraction attempts and analyze failure patterns."""
    
    def __init__(self):
        self.records = []
    
    def record(self, document_id: str, attempts: int, success: bool,
               errors: list[str], detected_pattern: str | None = None):
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---
    
    def summary(self) -> dict:
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---


print("Class defined.")

In [ ]:
# ==========================
# SOLUTION (TODO 6)
# ==========================

class ExtractionTracker:
    """Track extraction attempts and analyze failure patterns.
    
    Use this to identify systematic issues across many extractions
    and generate actionable recommendations.
    """
    
    def __init__(self):
        self.records = []
    
    def record(self, document_id: str, attempts: int, success: bool,
               errors: list[str], detected_pattern: str | None = None):
        """Record the result of an extraction attempt."""
        self.records.append({
            "document_id": document_id,
            "attempts": attempts,
            "success": success,
            "errors": errors,
            "detected_pattern": detected_pattern
        })
    
    def summary(self) -> dict:
        """Generate a summary with statistics and recommendations."""
        if not self.records:
            return {"total_extractions": 0, "message": "No extractions recorded."}
        
        total = len(self.records)
        successes = sum(1 for r in self.records if r["success"])
        failures = total - successes
        success_rate = successes / total
        avg_attempts = sum(r["attempts"] for r in self.records) / total
        
        # Count patterns (only from failures)
        pattern_counts = {}
        for r in self.records:
            if not r["success"] and r["detected_pattern"]:
                p = r["detected_pattern"]
                pattern_counts[p] = pattern_counts.get(p, 0) + 1
        
        # Generate recommendations
        recommendations = []
        
        # Check for dominant patterns
        if failures > 0:
            for pattern, count in sorted(pattern_counts.items(),
                                         key=lambda x: x[1], reverse=True):
                if count / failures > 0.30:
                    recommendations.append(
                        f"Add few-shot example for pattern '{pattern}' "
                        f"({count}/{failures} failures, {count/failures:.0%})"
                    )
        
        # Check retry rate
        if avg_attempts > 1.5:
            recommendations.append(
                f"High retry rate ({avg_attempts:.1f} avg attempts). "
                f"Review prompt for systematic issues."
            )
        
        # Check success rate
        if success_rate < 0.80:
            recommendations.append(
                f"Low success rate ({success_rate:.0%}). "
                f"Consider adding more specific tool descriptions."
            )
        
        if not recommendations:
            recommendations.append("System performing well. No immediate action needed.")
        
        return {
            "total_extractions": total,
            "success_count": successes,
            "failure_count": failures,
            "success_rate": success_rate,
            "avg_attempts": avg_attempts,
            "pattern_counts": pattern_counts,
            "recommendations": recommendations
        }


print("Solution loaded.")

In [ ]:
# ============================================================
# Run 10 simulated extractions and analyze patterns
# ============================================================

import random
random.seed(42)  # Reproducible results

tracker = ExtractionTracker()

# Simulate 10 extractions with realistic failure patterns
simulated_results = [
    # (doc_id, attempts, success, errors, pattern)
    ("doc_001", 1, True,  [], None),
    ("doc_002", 2, True,  ["Line items sum mismatch"], "math_error_line_items"),
    ("doc_003", 1, True,  [], None),
    ("doc_004", 3, False, ["vendor_name is empty"], "missing_vendor"),
    ("doc_005", 1, True,  [], None),
    ("doc_006", 2, True,  ["Line items sum mismatch"], "math_error_line_items"),
    ("doc_007", 1, True,  [], None),
    ("doc_008", 3, False, ["Line items sum mismatch"], "math_error_line_items"),
    ("doc_009", 2, True,  ["Date format wrong"], "date_format"),
    ("doc_010", 3, False, ["Line items sum mismatch"], "math_error_line_items"),
]

for doc_id, attempts, success, errors, pattern in simulated_results:
    tracker.record(doc_id, attempts, success, errors, pattern)

# Print the analysis
report = tracker.summary()

print("Extraction Performance Report")
print("=" * 50)
print(f"Total extractions:  {report['total_extractions']}")
print(f"Successes:          {report['success_count']}")
print(f"Failures:           {report['failure_count']}")
print(f"Success rate:       {report['success_rate']:.0%}")
print(f"Avg attempts:       {report['avg_attempts']:.1f}")

print(f"\nFailure Patterns:")
for pattern, count in sorted(report['pattern_counts'].items(),
                              key=lambda x: x[1], reverse=True):
    print(f"  {pattern}: {count} occurrences")

print(f"\nRecommendations:")
for rec in report['recommendations']:
    print(f"  -> {rec}")

In [ ]:
#@title 🎧 Listen: Todo6 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_todo6_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Reading the Report

Notice the recommendation: the `math_error_line_items` pattern dominates our failures. In practice, this tells us to:

1. **Add a few-shot example** showing correct line-item arithmetic
2. **Add a system prompt instruction**: "Double-check that each line item's qty * unit_price equals line_total before returning"
3. **Consider adding a `_calculation_check` field** to the schema that forces Claude to show its arithmetic

This is the feedback loop that turns a decent extraction system into a great one.

In [ ]:
#@title 🎧 Listen: Todo7
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_todo7.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Integration Exercise: Smart Extraction Pipeline

Time to put it all together. We will build a `SmartExtractionPipeline` that combines everything from this notebook:

1. **Tool-based extraction** (from Notebook 02)
2. **Semantic validation** (Section 1)
3. **Error classification** (Section 3)
4. **Smart retry** — only retry if errors are retryable (Section 3)
5. **Pattern tracking** (Section 4)

This is the kind of system you would deploy in production—and the kind of architecture the exam expects you to understand.

In [ ]:
# ============================================================
# TODO 7: Build a SmartExtractionPipeline
# ============================================================
#
# Combine all components into one cohesive pipeline:
#
# class SmartExtractionPipeline:
#     def __init__(self, client, tool, validator_fn, max_retries=2):
#         ...
#
#     def _detect_pattern(self, errors) -> str | None:
#         """Categorize the dominant error pattern."""
#         Look at the errors and return a pattern string like:
#         - "math_error_line_items" if any error mentions sum/total mismatch
#         - "math_error_unit_calc" if any error mentions qty * unit_price
#         - "format_error_date" if any error mentions date
#         - "format_error_invoice_number" if any error mentions invoice_number pattern
#         - "missing_field" if any error mentions empty/null/missing
#         - None if no clear pattern
#
#     def extract(self, document_id, document_text) -> dict:
#         """Full extraction with smart retry and tracking."""
#         1. Call API with tool_use
#         2. Validate the result
#         3. If errors:
#            a. Classify each error
#            b. Separate retryable from non-retryable
#            c. Only retry if there are retryable errors
#            d. In retry feedback, only mention retryable errors
#         4. Track the result (including detected pattern)
#         5. Return {
#              "document_id": str,
#              "data": dict,
#              "success": bool,
#              "attempts": int,
#              "retryable_errors": list[str],
#              "non_retryable_errors": list[str],
#              "detected_pattern": str | None
#            }
#
#     def report(self) -> dict:
#         """Return the tracker summary."""
# ============================================================

class SmartExtractionPipeline:
    """Production-grade extraction with validation, smart retry, and tracking."""
    
    def __init__(self, client, tool, validator_fn, max_retries=2):
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---
    
    def _detect_pattern(self, errors: list[str]) -> str | None:
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---
    
    def extract(self, document_id: str, document_text: str) -> dict:
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---
    
    def report(self) -> dict:
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---


print("Class defined.")

In [ ]:
# ==========================
# SOLUTION (TODO 7)
# ==========================

class SmartExtractionPipeline:
    """Production-grade extraction with validation, smart retry, and tracking.
    
    Combines:
    - Tool-based extraction
    - Semantic validation
    - Error classification (retryable vs futile)
    - Smart retry (only retries retryable errors)
    - Pattern tracking for systematic improvement
    """
    
    def __init__(self, client, tool: dict, validator_fn, max_retries: int = 2):
        self.client = client
        self.tool = tool
        self.validator_fn = validator_fn
        self.max_retries = max_retries
        self.tracker = ExtractionTracker()
    
    def _detect_pattern(self, errors: list[str]) -> str | None:
        """Categorize the dominant error pattern from a list of errors."""
        if not errors:
            return None
        
        combined = " ".join(errors).lower()
        
        if "sum" in combined and "total" in combined:
            return "math_error_line_items"
        if "qty" in combined and "unit_price" in combined:
            return "math_error_unit_calc"
        if "date" in combined and ("invalid" in combined or "future" in combined):
            return "format_error_date"
        if "invoice_number" in combined and ("pattern" in combined or "match" in combined):
            return "format_error_invoice_number"
        if "empty" in combined or "null" in combined or "missing" in combined:
            return "missing_field"
        
        return None
    
    def extract(self, document_id: str, document_text: str) -> dict:
        """Full extraction with smart retry and tracking."""
        messages = [
            {
                "role": "user",
                "content": f"Extract the invoice data from this document:\n\n{document_text}"
            }
        ]
        
        attempts = 0
        last_data = None
        all_retryable_errors = []
        all_non_retryable_errors = []
        
        for attempt in range(1 + self.max_retries):
            attempts += 1
            
            # Call API
            response = self.client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=1024,
                tools=[self.tool],
                tool_choice={"type": "tool", "name": self.tool["name"]},
                messages=messages
            )
            
            tool_block = response.content[0]
            extracted = tool_block.input
            last_data = extracted
            
            # Validate
            errors = self.validator_fn(extracted)
            
            if not errors:
                # Success!
                pattern = self._detect_pattern(all_retryable_errors)
                self.tracker.record(document_id, attempts, True, [], pattern)
                return {
                    "document_id": document_id,
                    "data": extracted,
                    "success": True,
                    "attempts": attempts,
                    "retryable_errors": [],
                    "non_retryable_errors": [],
                    "detected_pattern": pattern
                }
            
            # Classify errors
            retryable = [e for e in errors if classify_error(e) == "retryable"]
            non_retryable = [e for e in errors if classify_error(e) != "retryable"]
            
            all_retryable_errors = retryable
            all_non_retryable_errors = non_retryable
            
            # Only retry if there are retryable errors AND retries left
            if not retryable or attempt >= self.max_retries:
                break
            
            # Build retry feedback (only mention retryable errors)
            messages.append({
                "role": "assistant",
                "content": [{
                    "type": "tool_use",
                    "id": tool_block.id,
                    "name": tool_block.name,
                    "input": extracted
                }]
            })
            
            error_list = "\n".join(
                f"{i+1}. {err}" for i, err in enumerate(retryable)
            )
            feedback = (
                f"Your extraction had {len(retryable)} correctable error(s):\n"
                f"{error_list}\n"
                f"Please re-extract with these errors corrected. "
                f"Other fields were fine\u2014keep them as-is."
            )
            
            messages.append({
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": tool_block.id,
                    "content": feedback
                }]
            })
        
        # Exhausted retries or no retryable errors
        all_errors = all_retryable_errors + all_non_retryable_errors
        pattern = self._detect_pattern(all_errors)
        self.tracker.record(document_id, attempts, False, all_errors, pattern)
        
        return {
            "document_id": document_id,
            "data": last_data,
            "success": False,
            "attempts": attempts,
            "retryable_errors": all_retryable_errors,
            "non_retryable_errors": all_non_retryable_errors,
            "detected_pattern": pattern
        }
    
    def report(self) -> dict:
        """Return the tracker summary."""
        return self.tracker.summary()


print("Solution loaded.")

In [ ]:
# ============================================================
# Process 5 test documents through the pipeline
# ============================================================

pipeline = SmartExtractionPipeline(
    client=client,
    tool=INVOICE_TOOL,
    validator_fn=validate_invoice,
    max_retries=2
)

# Test documents with pre-configured mock responses
test_cases = [
    {
        "doc_id": "doc_001",
        "text": "Invoice from Acme Corp, INV-1001, 2025-02-15, 2x Widget @ $50 = $100. Total: $100",
        "responses": [
            {  # Correct on first try
                "vendor_name": "Acme Corp",
                "invoice_number": "INV-1001",
                "invoice_date": "2025-02-15",
                "total_amount": 100.00,
                "line_items": [{"description": "Widget", "qty": 2, "unit_price": 50.00, "line_total": 100.00}]
            }
        ]
    },
    {
        "doc_id": "doc_002",
        "text": "Invoice from BetaCo, INV-2002, 2025-01-20, 3x Gadget @ $25 = $75. Total: $75",
        "responses": [
            {  # Math error: wrong total
                "vendor_name": "BetaCo",
                "invoice_number": "INV-2002",
                "invoice_date": "2025-01-20",
                "total_amount": 100.00,  # WRONG
                "line_items": [{"description": "Gadget", "qty": 3, "unit_price": 25.00, "line_total": 75.00}]
            },
            {  # Corrected on retry
                "vendor_name": "BetaCo",
                "invoice_number": "INV-2002",
                "invoice_date": "2025-01-20",
                "total_amount": 75.00,
                "line_items": [{"description": "Gadget", "qty": 3, "unit_price": 25.00, "line_total": 75.00}]
            }
        ]
    },
    {
        "doc_id": "doc_003",
        "text": "Receipt from unknown vendor, no invoice number, 2025-03-10, 1x Service = $500. Total: $500",
        "responses": [
            {  # Missing vendor + bad invoice number (futile errors)
                "vendor_name": "",
                "invoice_number": "NONE",
                "invoice_date": "2025-03-10",
                "total_amount": 500.00,
                "line_items": [{"description": "Service", "qty": 1, "unit_price": 500.00, "line_total": 500.00}]
            }
        ]
    },
    {
        "doc_id": "doc_004",
        "text": "Invoice from GammaTech, INV-4004, 2025-02-28, 5x Module @ $120 = $600, 2x Cable @ $15 = $30. Total: $630",
        "responses": [
            {  # Line item calc error
                "vendor_name": "GammaTech",
                "invoice_number": "INV-4004",
                "invoice_date": "2025-02-28",
                "total_amount": 630.00,
                "line_items": [
                    {"description": "Module", "qty": 5, "unit_price": 120.00, "line_total": 500.00},  # WRONG: 5*120=600
                    {"description": "Cable", "qty": 2, "unit_price": 15.00, "line_total": 30.00}
                ]
            },
            {  # Still wrong total after fixing line item
                "vendor_name": "GammaTech",
                "invoice_number": "INV-4004",
                "invoice_date": "2025-02-28",
                "total_amount": 630.00,
                "line_items": [
                    {"description": "Module", "qty": 5, "unit_price": 120.00, "line_total": 600.00},
                    {"description": "Cable", "qty": 2, "unit_price": 15.00, "line_total": 30.00}
                ]
            }
        ]
    },
    {
        "doc_id": "doc_005",
        "text": "Invoice from DeltaSupply, INV-5005, 2025-01-05, 10x Part @ $8.50 = $85. Total: $85",
        "responses": [
            {  # Correct on first try
                "vendor_name": "DeltaSupply",
                "invoice_number": "INV-5005",
                "invoice_date": "2025-01-05",
                "total_amount": 85.00,
                "line_items": [{"description": "Part", "qty": 10, "unit_price": 8.50, "line_total": 85.00}]
            }
        ]
    }
]

print("Processing 5 documents through SmartExtractionPipeline")
print("=" * 60)

results = []
for tc in test_cases:
    client.set_responses(tc["responses"])
    result = pipeline.extract(tc["doc_id"], tc["text"])
    results.append(result)
    
    status = "SUCCESS" if result["success"] else "FAILED"
    print(f"\n[{status}] {result['document_id']} | Attempts: {result['attempts']}")
    
    if result["retryable_errors"]:
        print(f"  Retryable errors: {result['retryable_errors']}")
    if result["non_retryable_errors"]:
        print(f"  Non-retryable errors: {result['non_retryable_errors']}")
    if result["detected_pattern"]:
        print(f"  Pattern: {result['detected_pattern']}")

# Print the final report
print("\n" + "=" * 60)
print("PIPELINE REPORT")
print("=" * 60)

report = pipeline.report()
print(f"Total extractions:  {report['total_extractions']}")
print(f"Success rate:       {report['success_rate']:.0%}")
print(f"Avg attempts:       {report['avg_attempts']:.1f}")

if report.get('pattern_counts'):
    print(f"\nFailure Patterns:")
    for pattern, count in report['pattern_counts'].items():
        print(f"  {pattern}: {count}")

print(f"\nRecommendations:")
for rec in report['recommendations']:
    print(f"  -> {rec}")

In [ ]:
#@title 🎧 Listen: Todo7 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_todo7_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### What Happened

Let's walk through what the pipeline did for each document:

| Document | Attempt 1 | What Happened | Result |
|---|---|---|---|
| doc_001 | Correct | No errors, returned immediately | SUCCESS (1 attempt) |
| doc_002 | Wrong total | Retryable math error, retried with feedback, fixed | SUCCESS (2 attempts) |
| doc_003 | Missing vendor | Non-retryable (futile), did NOT waste retries | FAILED (1 attempt) |
| doc_004 | Wrong line calc | Retryable, fixed line item on retry, total now correct | SUCCESS (2 attempts) |
| doc_005 | Correct | No errors, returned immediately | SUCCESS (1 attempt) |

Notice doc_003: the pipeline recognized that "vendor_name is empty" and "invoice_number doesn't match INV-XXXX" are **non-retryable** errors. Instead of wasting 2 retries, it returned immediately with a clear explanation of why it failed. This is the smart part of smart retry.

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Key Takeaways

Here is what you need to remember for the exam and for production systems:

1. **Validation catches semantic errors that schemas miss.** A JSON schema ensures correct types; validation ensures correct values. Line items summing correctly, dates being logical, business rules being satisfied—these all require post-extraction validation.

2. **Retry-with-error-feedback: append errors to the conversation for targeted correction.** Don't just say "try again."​ Feed Claude the specific errors from its previous attempt so it can fix exactly what went wrong without re-introducing new issues.

3. **Never retry for missing information.** If a field is null because the data isn't in the source document, retrying will either return null again or hallucinate. Use nullable fields in your schema and handle null downstream.

4. **Classify errors before deciding to retry.** Retryable (math/format) errors are worth retrying. Futile (missing info) errors are not. Ambiguous errors should be flagged for human review.

5. **Track `detected_pattern` for systematic failure analysis.** If >30% of failures share a pattern, add a few-shot example targeting that pattern. This is how you continuously improve extraction accuracy.

6. **Smart retry saves tokens and prevents hallucination.** By only retrying retryable errors, you avoid wasting API calls and—more importantly—avoid pushing Claude to fabricate data just to satisfy a validation constraint it can never meet.

---

## What's Next

In **Notebook 4: Batch API & Multi-Pass Review**, we will tackle:
- Structuring batch requests with `custom_id` for large-scale processing
- Multi-pass review architectures where different passes catch different error types
- Combining everything into a production-grade document processing system

The retry loop you built here will plug directly into the batch pipeline—each document in the batch goes through the same extract-validate-retry cycle.

---

*Built with Vizuara — AI education that meets you where you are.*